In [1]:
# Cell 1: Imports and configuration
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader

# Configuration - all key settings in one place
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

# Urgency labels in severity order
URGENCY_LABELS = ["Low", "Medium", "High", "Critical"]
label2id = {label: idx for idx, label in enumerate(URGENCY_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"\nModel: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}")
print(f"Labels: {URGENCY_LABELS}")

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti

Model: distilbert-base-uncased
Epochs: 10, Batch size: 16, Learning rate: 3e-05
Labels: ['Low', 'Medium', 'High', 'Critical']


In [2]:
# Cell 2: Load data and create train/validation split
DATA_PATH = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_synthetic_stage2b.csv"

df = pd.read_csv(DATA_PATH)
print(f"Total records: {len(df)}")

# Stratified split to keep urgency distribution balanced across train and validation
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df["urgency_label"],
)

print(f"Training set: {len(train_df)} records")
print(f"Validation set: {len(val_df)} records")

print("\nTraining set urgency distribution:")
print(train_df["urgency_label"].value_counts().sort_index())

print("\nValidation set urgency distribution:")
print(val_df["urgency_label"].value_counts().sort_index())

Total records: 1000
Training set: 800 records
Validation set: 200 records

Training set urgency distribution:
urgency_label
Critical    173
High        209
Low         146
Medium      272
Name: count, dtype: int64

Validation set urgency distribution:
urgency_label
Critical    44
High        52
Low         36
Medium      68
Name: count, dtype: int64


In [3]:
# Cell 3: Dataset class and tokeniser
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SafeguardingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, max_length=MAX_LENGTH):
        self.texts = dataframe["free_text"].tolist()
        self.labels = [label2id[label] for label in dataframe["urgency_label"]]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Create datasets and dataloaders
train_dataset = SafeguardingDataset(train_df, tokenizer, label2id)
val_dataset = SafeguardingDataset(val_df, tokenizer, label2id)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Tokeniser loaded: {MODEL_NAME}")
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

Tokeniser loaded: distilbert-base-uncased
Train batches: 50, Validation batches: 13


In [4]:
# Cell 4: Load model and configure training
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(URGENCY_LABELS),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Model loaded and moved to {device}")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and moved to cuda
Total training steps: 500
Warmup steps: 50
Trainable parameters: 66,956,548


In [5]:
# Cell 5: Training loop with per-epoch evaluation
train_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    # --- Training phase ---
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation phase ---
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    val_acc = correct / len(all_labels)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}  |  Train loss: {avg_train_loss:.4f}  |  Val accuracy: {val_acc:.4f}")

print("\nTraining complete.")

Epoch 1/10  |  Train loss: 1.2332  |  Val accuracy: 0.9400
Epoch 2/10  |  Train loss: 0.1498  |  Val accuracy: 1.0000
Epoch 3/10  |  Train loss: 0.0115  |  Val accuracy: 1.0000
Epoch 4/10  |  Train loss: 0.0063  |  Val accuracy: 1.0000
Epoch 5/10  |  Train loss: 0.0042  |  Val accuracy: 1.0000
Epoch 6/10  |  Train loss: 0.0034  |  Val accuracy: 1.0000
Epoch 7/10  |  Train loss: 0.0028  |  Val accuracy: 1.0000
Epoch 8/10  |  Train loss: 0.0024  |  Val accuracy: 1.0000
Epoch 9/10  |  Train loss: 0.0022  |  Val accuracy: 1.0000
Epoch 10/10  |  Train loss: 0.0021  |  Val accuracy: 1.0000

Training complete.


In [6]:
# Cell 6: Detailed classification report on validation set
print("Classification Report on Validation Set:\n")
print(classification_report(
    all_labels,
    all_preds,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm, index=URGENCY_LABELS, columns=URGENCY_LABELS)
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"
print(cm_df)

Classification Report on Validation Set:

              precision    recall  f1-score   support

         Low     1.0000    1.0000    1.0000        36
      Medium     1.0000    1.0000    1.0000        68
        High     1.0000    1.0000    1.0000        52
    Critical     1.0000    1.0000    1.0000        44

    accuracy                         1.0000       200
   macro avg     1.0000    1.0000    1.0000       200
weighted avg     1.0000    1.0000    1.0000       200


Confusion Matrix:
Predicted  Low  Medium  High  Critical
Actual                                
Low         36       0     0         0
Medium       0      68     0         0
High         0       0    52         0
Critical     0       0     0        44


In [7]:
# Cell 7: Test with hand-written examples the model has never seen
test_examples = [
    {
        "text": "A cadet mentioned that they sometimes don't have breakfast before coming to parade because there isn't much food at home. They seemed a bit tired but otherwise engaged in activities normally.",
        "expected": "Low",
    },
    {
        "text": "During a weekend exercise, a cadet was seen pushing another cadet repeatedly and calling them names in front of the section. The targeted cadet became visibly upset and withdrew from the group for the rest of the evening.",
        "expected": "Medium",
    },
    {
        "text": "A cadet disclosed to their section commander that they have been cutting themselves on their arms and showed recent marks. They asked the adult not to tell anyone. The cadet appeared distressed and said things at home are getting worse.",
        "expected": "High",
    },
    {
        "text": "A cadet reported that an adult instructor touched them inappropriately in the stores area after the other cadets had left. The cadet was shaking and crying when they told the duty officer. This is the second time a concern about this instructor has been raised.",
        "expected": "Critical",
    },
]

model.eval()
print("Testing model on unseen hand-written examples:\n")

for i, example in enumerate(test_examples):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
        probs = torch.softmax(outputs.logits, dim=-1).squeeze()

    predicted_id = torch.argmax(probs).item()
    predicted_label = id2label[predicted_id]
    expected_label = example["expected"]
    match = "CORRECT" if predicted_label == expected_label else "WRONG"

    print(f"Example {i+1}: Expected={expected_label}, Predicted={predicted_label} [{match}]")
    for label, prob in zip(URGENCY_LABELS, probs.tolist()):
        bar = "█" * int(prob * 40)
        print(f"  {label:>8}: {prob:.4f} {bar}")
    print()

Testing model on unseen hand-written examples:

Example 1: Expected=Low, Predicted=Low [CORRECT]
       Low: 0.9900 ███████████████████████████████████████
    Medium: 0.0058 
      High: 0.0012 
  Critical: 0.0029 

Example 2: Expected=Medium, Predicted=Low [WRONG]
       Low: 0.9850 ███████████████████████████████████████
    Medium: 0.0123 
      High: 0.0013 
  Critical: 0.0014 

Example 3: Expected=High, Predicted=Low [WRONG]
       Low: 0.9818 ███████████████████████████████████████
    Medium: 0.0134 
      High: 0.0015 
  Critical: 0.0032 

Example 4: Expected=Critical, Predicted=Low [WRONG]
       Low: 0.9799 ███████████████████████████████████████
    Medium: 0.0167 
      High: 0.0015 
  Critical: 0.0019 



In [8]:
# Cell 8: Inspect synthetic data patterns by urgency band
for label in URGENCY_LABELS:
    subset = df[df["urgency_label"] == label]["free_text"]
    sample = subset.sample(2, random_state=42)
    print(f"=== {label.upper()} ===")
    for text in sample:
        print(f"  {text[:200]}...")
        print()

=== LOW ===
  Cadet hinted that they are being pressured to go to addresses with people they do not know well and feel unable to say no. At present there is no indication of immediate risk, but the situation will b...

  Cadet showed staff messages received on social media containing threats and humiliating comments from peers. The concern appears low-level at this stage; staff will continue to observe and provide a s...

=== MEDIUM ===
  Cadet reported being added to group chats where explicit or disturbing content is shared, and feels pressured to remain in them. Further conversation with the cadet and parents/carers is needed to und...

  Cadet, who was previously enthusiastic, has begun arriving late, leaving early and avoiding taking part in team tasks. Staff have agreed to monitor the situation closely and to share information with ...

=== HIGH ===
  Another cadet reported that this cadet has spoken about wanting to travel to conflict zones and ‘fight for a cause’. The informati

In [9]:
# Cell 9: Examine where the action/response language starts
# Let's look at a few full examples to find the pattern

for label in URGENCY_LABELS:
    subset = df[df["urgency_label"] == label]["free_text"]
    sample = subset.sample(3, random_state=99)
    print(f"\n=== {label.upper()} ===")
    for text in sample:
        print(f"\n  {text}")
        print(f"  ---")


=== LOW ===

  Cadet reports repeated unkind comments and name-calling from a small group during parade nights. At present there is no indication of immediate risk, but the situation will be monitored and support options explained to the cadet.
  ---

  Another cadet reported that this cadet has spoken about wanting to travel to conflict zones and ‘fight for a cause’. No urgent safeguarding action is required currently, though the information has been recorded and will be reviewed periodically.
  ---

  Cadet hinted that there are ongoing problems between carers at home and that they often stay out late to avoid being there. No urgent safeguarding action is required currently, though the information has been recorded and will be reviewed periodically. The cadet has attended most recent parades but staff have noticed changes in presentation and demeanour.
  ---

=== MEDIUM ===

  Staff were told that the cadet has been missing from home overnight on several occasions and returns with u

In [10]:
# Cell 9: Retrain on v2 dataset
# Load the improved synthetic data
DATA_PATH_V2 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v2.csv"
df_v2 = pd.read_csv(DATA_PATH_V2)
print(f"Loaded v2 dataset: {len(df_v2)} records")

# Fresh stratified split
train_df_v2, val_df_v2 = train_test_split(
    df_v2,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_v2["urgency_label"],
)
print(f"Training set: {len(train_df_v2)}")
print(f"Validation set: {len(val_df_v2)}")

# Create datasets and loaders
train_dataset_v2 = SafeguardingDataset(train_df_v2, tokenizer, label2id)
val_dataset_v2 = SafeguardingDataset(val_df_v2, tokenizer, label2id)
train_loader_v2 = DataLoader(train_dataset_v2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_v2 = DataLoader(val_dataset_v2, batch_size=BATCH_SIZE, shuffle=False)

# Load a FRESH model - not the one we already trained
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(URGENCY_LABELS),
    id2label=id2label,
    label2id=label2id,
)
model_v2.to(device)

optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LEARNING_RATE)
total_steps_v2 = len(train_loader_v2) * EPOCHS
warmup_steps_v2 = int(total_steps_v2 * WARMUP_RATIO)
scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2,
    num_warmup_steps=warmup_steps_v2,
    num_training_steps=total_steps_v2,
)

print(f"\nFresh model loaded on {device}")
print(f"Training batches per epoch: {len(train_loader_v2)}")
print(f"Total steps: {total_steps_v2}, Warmup: {warmup_steps_v2}")

Loaded v2 dataset: 1200 records
Training set: 960
Validation set: 240


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Fresh model loaded on cuda
Training batches per epoch: 60
Total steps: 600, Warmup: 60


In [11]:
# Cell 10: Train on v2 data
train_losses_v2 = []
val_accuracies_v2 = []

for epoch in range(EPOCHS):
    model_v2.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader_v2):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_v2.zero_grad()
        outputs = model_v2(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), max_norm=1.0)
        optimizer_v2.step()
        scheduler_v2.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader_v2)
    train_losses_v2.append(avg_train_loss)

    # Validation
    model_v2.eval()
    all_preds_v2 = []
    all_labels_v2 = []

    with torch.no_grad():
        for batch in val_loader_v2:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_v2(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds_v2.extend(preds.cpu().tolist())
            all_labels_v2.extend(labels.cpu().tolist())

    correct = sum(p == l for p, l in zip(all_preds_v2, all_labels_v2))
    val_acc = correct / len(all_labels_v2)
    val_accuracies_v2.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}  |  Train loss: {avg_train_loss:.4f}  |  Val accuracy: {val_acc:.4f}")

print("\nTraining complete.")

Epoch 1/10  |  Train loss: 1.3490  |  Val accuracy: 0.5875
Epoch 2/10  |  Train loss: 0.8297  |  Val accuracy: 0.6875
Epoch 3/10  |  Train loss: 0.5365  |  Val accuracy: 0.6625
Epoch 4/10  |  Train loss: 0.3966  |  Val accuracy: 0.7042
Epoch 5/10  |  Train loss: 0.2654  |  Val accuracy: 0.6792
Epoch 6/10  |  Train loss: 0.1761  |  Val accuracy: 0.7083
Epoch 7/10  |  Train loss: 0.1157  |  Val accuracy: 0.7042
Epoch 8/10  |  Train loss: 0.0695  |  Val accuracy: 0.7208
Epoch 9/10  |  Train loss: 0.0502  |  Val accuracy: 0.7208
Epoch 10/10  |  Train loss: 0.0425  |  Val accuracy: 0.7125

Training complete.


In [12]:
# Cell 11: Detailed classification report on v2 model
print("Classification Report (v2 model on v2 data):\n")
print(classification_report(
    all_labels_v2,
    all_preds_v2,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels_v2, all_preds_v2)
cm_df = pd.DataFrame(cm, index=URGENCY_LABELS, columns=URGENCY_LABELS)
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"
print(cm_df)

Classification Report (v2 model on v2 data):

              precision    recall  f1-score   support

         Low     0.9615    0.8333    0.8929        60
      Medium     0.7460    0.7833    0.7642        60
        High     0.5333    0.5333    0.5333        60
    Critical     0.6462    0.7000    0.6720        60

    accuracy                         0.7125       240
   macro avg     0.7218    0.7125    0.7156       240
weighted avg     0.7218    0.7125    0.7156       240


Confusion Matrix:
Predicted  Low  Medium  High  Critical
Actual                                
Low         50      10     0         0
Medium       2      47    11         0
High         0       5    32        23
Critical     0       1    17        42


In [13]:
# Cell 12: Test v2 model with hand-written examples
test_examples = [
    {
        "text": "A cadet mentioned that they sometimes don't have breakfast before coming to parade because there isn't much food at home. They seemed a bit tired but otherwise engaged in activities normally.",
        "expected": "Low",
    },
    {
        "text": "During a weekend exercise, a cadet was seen pushing another cadet repeatedly and calling them names in front of the section. The targeted cadet became visibly upset and withdrew from the group for the rest of the evening.",
        "expected": "Medium",
    },
    {
        "text": "A cadet disclosed to their section commander that they have been cutting themselves on their arms and showed recent marks. They asked the adult not to tell anyone. The cadet appeared distressed and said things at home are getting worse.",
        "expected": "High",
    },
    {
        "text": "A cadet reported that an adult instructor touched them inappropriately in the stores area after the other cadets had left. The cadet was shaking and crying when they told the duty officer. This is the second time a concern about this instructor has been raised.",
        "expected": "Critical",
    },
]

model_v2.eval()
print("Testing v2 model on unseen hand-written examples:\n")

for i, example in enumerate(test_examples):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model_v2(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
        probs = torch.softmax(outputs.logits, dim=-1).squeeze()

    predicted_id = torch.argmax(probs).item()
    predicted_label = id2label[predicted_id]
    expected_label = example["expected"]
    match = "CORRECT" if predicted_label == expected_label else "WRONG"

    print(f"Example {i+1}: Expected={expected_label}, Predicted={predicted_label} [{match}]")
    for label, prob in zip(URGENCY_LABELS, probs.tolist()):
        bar = "█" * int(prob * 40)
        print(f"  {label:>8}: {prob:.4f} {bar}")
    print()

Testing v2 model on unseen hand-written examples:

Example 1: Expected=Low, Predicted=Low [CORRECT]
       Low: 0.9950 ███████████████████████████████████████
    Medium: 0.0014 
      High: 0.0018 
  Critical: 0.0018 

Example 2: Expected=Medium, Predicted=Medium [CORRECT]
       Low: 0.0026 
    Medium: 0.9689 ██████████████████████████████████████
      High: 0.0274 █
  Critical: 0.0012 

Example 3: Expected=High, Predicted=Critical [WRONG]
       Low: 0.0032 
    Medium: 0.0032 
      High: 0.1436 █████
  Critical: 0.8501 ██████████████████████████████████

Example 4: Expected=Critical, Predicted=Medium [WRONG]
       Low: 0.0033 
    Medium: 0.9356 █████████████████████████████████████
      High: 0.0583 ██
  Critical: 0.0029 



In [14]:
# Cell 13: Check how the model performs specifically on abuse-related categories
abuse_val = val_df_v2[val_df_v2["category_label"] == "Abuse by adult in organisation"]
print(f"Abuse cases in validation set: {len(abuse_val)}")
print(f"Urgency distribution:")
print(abuse_val["urgency_label"].value_counts().sort_index())

# Test model on these specific cases
model_v2.eval()
abuse_preds = []
abuse_true = []

for _, row in abuse_val.iterrows():
    encoded = tokenizer(
        row["free_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)
    
    with torch.no_grad():
        outputs = model_v2(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    abuse_preds.append(pred)
    abuse_true.append(label2id[row["urgency_label"]])

print("\nClassification report for 'Abuse by adult in organisation' only:\n")
print(classification_report(
    abuse_true,
    abuse_preds,
    target_names=URGENCY_LABELS,
    digits=4,
))

Abuse cases in validation set: 28
Urgency distribution:
urgency_label
Critical    6
High        4
Low         9
Medium      9
Name: count, dtype: int64

Classification report for 'Abuse by adult in organisation' only:

              precision    recall  f1-score   support

         Low     0.8889    0.8889    0.8889         9
      Medium     0.8750    0.7778    0.8235         9
        High     0.3333    0.5000    0.4000         4
    Critical     0.6000    0.5000    0.5455         6

    accuracy                         0.7143        28
   macro avg     0.6743    0.6667    0.6645        28
weighted avg     0.7432    0.7143    0.7244        28



In [18]:
save_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_v2"
model_v2.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

Model saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\urgency_distilbert_v2


In [20]:
# Cell 15: Load v2 data and set up dual-task labels
DATA_PATH_V2 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v2.csv"
df_v2 = pd.read_csv(DATA_PATH_V2)

# Category labels from the dataset
CATEGORY_LABELS = sorted(df_v2["category_label"].unique().tolist())

category2id = {label: idx for idx, label in enumerate(CATEGORY_LABELS)}
id2category = {idx: label for label, idx in category2id.items()}

print(f"Loaded {len(df_v2)} records")
print(f"\nUrgency labels ({len(URGENCY_LABELS)}): {URGENCY_LABELS}")
print(f"\nCategory labels ({len(CATEGORY_LABELS)}):")
for i, cat in enumerate(CATEGORY_LABELS):
    print(f"  {i}: {cat}")

# Stratified split - stratify on urgency as before
train_df_mt, val_df_mt = train_test_split(
    df_v2,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_v2["urgency_label"],
)

print(f"\nTrain: {len(train_df_mt)}, Validation: {len(val_df_mt)}")

Loaded 1200 records

Urgency labels (4): ['Low', 'Medium', 'High', 'Critical']

Category labels (12):
  0: Abuse by adult in organisation
  1: Attendance / engagement
  2: Behaviour / conduct
  3: Bullying / peer conflict
  4: Exploitation / trafficking
  5: FGM / harmful practices
  6: Financial hardship
  7: Home issues / neglect
  8: Mental health / self-harm
  9: Online safety
  10: Physical safety
  11: Radicalisation / extremism

Train: 960, Validation: 240


In [21]:
# Cell 16: Multi-task dataset class
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, category2id, max_length=MAX_LENGTH):
        self.texts = dataframe["free_text"].tolist()
        self.urgency_labels = [label2id[label] for label in dataframe["urgency_label"]]
        self.category_labels = [category2id[label] for label in dataframe["category_label"]]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "urgency_label": torch.tensor(self.urgency_labels[idx], dtype=torch.long),
            "category_label": torch.tensor(self.category_labels[idx], dtype=torch.long),
        }

train_dataset_mt = MultiTaskDataset(train_df_mt, tokenizer, label2id, category2id)
val_dataset_mt = MultiTaskDataset(val_df_mt, tokenizer, label2id, category2id)

train_loader_mt = DataLoader(train_dataset_mt, batch_size=BATCH_SIZE, shuffle=True)
val_loader_mt = DataLoader(val_dataset_mt, batch_size=BATCH_SIZE, shuffle=False)

# Verify
sample = train_dataset_mt[0]
print(f"Sample urgency label: {sample['urgency_label'].item()} -> {id2label[sample['urgency_label'].item()]}")
print(f"Sample category label: {sample['category_label'].item()} -> {id2category[sample['category_label'].item()]}")
print(f"Train batches: {len(train_loader_mt)}, Val batches: {len(val_loader_mt)}")

Sample urgency label: 2 -> High
Sample category label: 1 -> Attendance / engagement
Train batches: 60, Val batches: 15


In [22]:
# Cell 17: Multi-task model definition
from transformers import DistilBertModel
import torch.nn as nn

class MultiTaskSafeguardingModel(nn.Module):
    def __init__(self, model_name, num_urgency_labels, num_category_labels):
        super().__init__()
        
        # Shared DistilBERT body
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size  # 768 for distilbert-base
        
        # Urgency classification head
        self.urgency_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_urgency_labels),
        )
        
        # Category classification head
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_category_labels),
        )

    def forward(self, input_ids, attention_mask):
        # Shared encoding
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use the [CLS] token representation (first token)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Two separate predictions
        urgency_logits = self.urgency_head(cls_output)
        category_logits = self.category_head(cls_output)
        
        return urgency_logits, category_logits

# Create the model
model_mt = MultiTaskSafeguardingModel(
    model_name=MODEL_NAME,
    num_urgency_labels=len(URGENCY_LABELS),
    num_category_labels=len(CATEGORY_LABELS),
)
model_mt.to(device)

total_params = sum(p.numel() for p in model_mt.parameters() if p.requires_grad)
print(f"Multi-task model loaded on {device}")
print(f"Trainable parameters: {total_params:,}")
print(f"Urgency classes: {len(URGENCY_LABELS)}")
print(f"Category classes: {len(CATEGORY_LABELS)}")

Multi-task model loaded on cuda
Trainable parameters: 67,556,368
Urgency classes: 4
Category classes: 12


In [24]:
# Cell 18: Train multi-task model
optimizer_mt = torch.optim.AdamW(model_mt.parameters(), lr=LEARNING_RATE)
total_steps_mt = len(train_loader_mt) * EPOCHS
warmup_steps_mt = int(total_steps_mt * WARMUP_RATIO)

scheduler_mt = get_linear_schedule_with_warmup(
    optimizer_mt,
    num_warmup_steps=warmup_steps_mt,
    num_training_steps=total_steps_mt,
)

loss_fn = nn.CrossEntropyLoss()

# Weight for combining the two losses
# 0.5/0.5 treats them equally - we can adjust if needed
URGENCY_WEIGHT = 0.5
CATEGORY_WEIGHT = 0.5

for epoch in range(EPOCHS):
    model_mt.train()
    running_loss = 0.0
    running_urg_loss = 0.0
    running_cat_loss = 0.0

    for batch in train_loader_mt:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        urgency_labels = batch["urgency_label"].to(device)
        category_labels = batch["category_label"].to(device)

        optimizer_mt.zero_grad()

        urgency_logits, category_logits = model_mt(input_ids, attention_mask)

        urg_loss = loss_fn(urgency_logits, urgency_labels)
        cat_loss = loss_fn(category_logits, category_labels)
        total_loss = (URGENCY_WEIGHT * urg_loss) + (CATEGORY_WEIGHT * cat_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mt.parameters(), max_norm=1.0)
        optimizer_mt.step()
        scheduler_mt.step()

        running_loss += total_loss.item()
        running_urg_loss += urg_loss.item()
        running_cat_loss += cat_loss.item()

    n_batches = len(train_loader_mt)

    # Validation
    model_mt.eval()
    urg_preds, urg_true = [], []
    cat_preds, cat_true = [], []

    with torch.no_grad():
        for batch in val_loader_mt:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            urgency_logits, category_logits = model_mt(input_ids, attention_mask)

            urg_preds.extend(torch.argmax(urgency_logits, dim=-1).cpu().tolist())
            urg_true.extend(batch["urgency_label"].tolist())
            cat_preds.extend(torch.argmax(category_logits, dim=-1).cpu().tolist())
            cat_true.extend(batch["category_label"].tolist())

    urg_acc = sum(p == l for p, l in zip(urg_preds, urg_true)) / len(urg_true)
    cat_acc = sum(p == l for p, l in zip(cat_preds, cat_true)) / len(cat_true)

    print(f"Epoch {epoch+1}/{EPOCHS}  |  Loss: {running_loss/n_batches:.4f} "
          f"(urg: {running_urg_loss/n_batches:.4f}, cat: {running_cat_loss/n_batches:.4f})  |  "
          f"Val urgency: {urg_acc:.4f}  Val category: {cat_acc:.4f}")

print("\nTraining complete.")

Epoch 1/10  |  Loss: 1.5928 (urg: 1.0871, cat: 2.0986)  |  Val urgency: 0.6250  Val category: 0.5083
Epoch 2/10  |  Loss: 1.1868 (urg: 0.8561, cat: 1.5175)  |  Val urgency: 0.6625  Val category: 0.6875
Epoch 3/10  |  Loss: 0.8197 (urg: 0.6618, cat: 0.9777)  |  Val urgency: 0.6708  Val category: 0.7542
Epoch 4/10  |  Loss: 0.5935 (urg: 0.5228, cat: 0.6642)  |  Val urgency: 0.7083  Val category: 0.7875
Epoch 5/10  |  Loss: 0.4339 (urg: 0.3921, cat: 0.4757)  |  Val urgency: 0.7292  Val category: 0.8458
Epoch 6/10  |  Loss: 0.3186 (urg: 0.2921, cat: 0.3451)  |  Val urgency: 0.7167  Val category: 0.8583
Epoch 7/10  |  Loss: 0.2389 (urg: 0.2207, cat: 0.2572)  |  Val urgency: 0.6875  Val category: 0.8583
Epoch 8/10  |  Loss: 0.1827 (urg: 0.1641, cat: 0.2014)  |  Val urgency: 0.7083  Val category: 0.8708
Epoch 9/10  |  Loss: 0.1438 (urg: 0.1234, cat: 0.1642)  |  Val urgency: 0.7083  Val category: 0.8708
Epoch 10/10  |  Loss: 0.1259 (urg: 0.1034, cat: 0.1484)  |  Val urgency: 0.7208  Val catego

In [25]:
# Cell 19: Detailed evaluation of multi-task model
print("=== URGENCY CLASSIFICATION ===\n")
print(classification_report(
    urg_true,
    urg_preds,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\n=== CATEGORY CLASSIFICATION ===\n")
print(classification_report(
    cat_true,
    cat_preds,
    target_names=CATEGORY_LABELS,
    digits=4,
))

=== URGENCY CLASSIFICATION ===

              precision    recall  f1-score   support

         Low     0.9444    0.8500    0.8947        60
      Medium     0.7143    0.8333    0.7692        60
        High     0.5556    0.5833    0.5691        60
    Critical     0.6981    0.6167    0.6549        60

    accuracy                         0.7208       240
   macro avg     0.7281    0.7208    0.7220       240
weighted avg     0.7281    0.7208    0.7220       240


=== CATEGORY CLASSIFICATION ===

                                precision    recall  f1-score   support

Abuse by adult in organisation     0.9333    1.0000    0.9655        28
       Attendance / engagement     0.8000    0.7407    0.7692        27
           Behaviour / conduct     0.7500    0.6250    0.6818        24
      Bullying / peer conflict     0.9565    0.8800    0.9167        25
    Exploitation / trafficking     0.9444    0.9444    0.9444        18
       FGM / harmful practices     1.0000    1.0000    1.0000     

In [26]:
# Cell 20: Test multi-task model on hand-written examples
import torch.nn.functional as F

test_examples = [
    {
        "text": "A cadet mentioned that they sometimes don't have breakfast before coming to parade because there isn't much food at home. They seemed a bit tired but otherwise engaged in activities normally.",
        "expected_urgency": "Low",
        "expected_category": "Home issues / neglect",
    },
    {
        "text": "During a weekend exercise, a cadet was seen pushing another cadet repeatedly and calling them names in front of the section. The targeted cadet became visibly upset and withdrew from the group for the rest of the evening.",
        "expected_urgency": "Medium",
        "expected_category": "Bullying / peer conflict",
    },
    {
        "text": "A cadet disclosed to their section commander that they have been cutting themselves on their arms and showed recent marks. They asked the adult not to tell anyone. The cadet appeared distressed and said things at home are getting worse.",
        "expected_urgency": "High",
        "expected_category": "Mental health / self-harm",
    },
    {
        "text": "A cadet reported that an adult instructor touched them inappropriately in the stores area after the other cadets had left. The cadet was shaking and crying when they told the duty officer. This is the second time a concern about this instructor has been raised.",
        "expected_urgency": "Critical",
        "expected_category": "Abuse by adult in organisation",
    },
    {
        "text": "A cadet told me that an older boy in the unit has been taking his phone at every parade night and going through his messages. The cadet said he's asked for it back but gets told to shut up. He looked embarrassed talking about it.",
        "expected_urgency": "Medium",
        "expected_category": "Bullying / peer conflict",
    },
]

model_mt.eval()
print("Multi-task model predictions on unseen examples:\n")

for i, ex in enumerate(test_examples):
    encoded = tokenizer(
        ex["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        urg_logits, cat_logits = model_mt(encoded["input_ids"], encoded["attention_mask"])
        urg_probs = F.softmax(urg_logits, dim=-1).squeeze()
        cat_probs = F.softmax(cat_logits, dim=-1).squeeze()

    pred_urg = id2label[torch.argmax(urg_probs).item()]
    pred_cat = id2category[torch.argmax(cat_probs).item()]
    cat_conf = torch.max(cat_probs).item()

    urg_match = "CORRECT" if pred_urg == ex["expected_urgency"] else "WRONG"
    cat_match = "CORRECT" if pred_cat == ex["expected_category"] else "WRONG"

    print(f"Example {i+1}:")
    print(f"  Urgency:  Expected={ex['expected_urgency']}, Predicted={pred_urg} [{urg_match}]")
    print(f"  Category: Expected={ex['expected_category']}, Predicted={pred_cat} ({cat_conf:.1%}) [{cat_match}]")
    print()

Multi-task model predictions on unseen examples:

Example 1:
  Urgency:  Expected=Low, Predicted=Low [CORRECT]
  Category: Expected=Home issues / neglect, Predicted=Home issues / neglect (89.2%) [CORRECT]

Example 2:
  Urgency:  Expected=Medium, Predicted=High [WRONG]
  Category: Expected=Bullying / peer conflict, Predicted=Behaviour / conduct (91.0%) [WRONG]

Example 3:
  Urgency:  Expected=High, Predicted=Critical [WRONG]
  Category: Expected=Mental health / self-harm, Predicted=Mental health / self-harm (79.4%) [CORRECT]

Example 4:
  Urgency:  Expected=Critical, Predicted=Critical [CORRECT]
  Category: Expected=Abuse by adult in organisation, Predicted=Abuse by adult in organisation (43.1%) [CORRECT]

Example 5:
  Urgency:  Expected=Medium, Predicted=High [WRONG]
  Category: Expected=Bullying / peer conflict, Predicted=Behaviour / conduct (39.8%) [WRONG]



In [29]:
# Cell 21: Save multi-task model
import os
save_path_mt = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v1"
os.makedirs(save_path_mt, exist_ok=True)

torch.save(model_mt.state_dict(), os.path.join(save_path_mt, "model_state.pt"))
tokenizer.save_pretrained(save_path_mt)

import json
config = {
    "model_name": MODEL_NAME,
    "num_urgency_labels": len(URGENCY_LABELS),
    "num_category_labels": len(CATEGORY_LABELS),
    "urgency_labels": URGENCY_LABELS,
    "category_labels": CATEGORY_LABELS,
    "label2id": label2id,
    "id2label": id2label,
    "category2id": category2id,
    "id2category": id2category,
    "max_length": MAX_LENGTH,
    "val_urgency_accuracy": 0.7208,
    "val_category_accuracy": 0.8792,
}
with open(os.path.join(save_path_mt, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to: {save_path_mt}")
print("Contents:")
for fn in os.listdir(save_path_mt):
    size = os.path.getsize(os.path.join(save_path_mt, fn))
    print(f"  {fn}: {size / 1024:.1f} KB")

Model saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v1
Contents:
  config.json: 1.7 KB
  model_state.pt: 263934.9 KB
  special_tokens_map.json: 0.1 KB
  tokenizer.json: 695.0 KB
  tokenizer_config.json: 1.3 KB
  vocab.txt: 226.1 KB


In [31]:
# Cell 22: Load v3 data and set up for multi-task retraining
DATA_PATH_V3 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v3.csv"
df_v3 = pd.read_csv(DATA_PATH_V3)

CATEGORY_LABELS_V3 = sorted(df_v3["category_label"].unique().tolist())
category2id_v3 = {label: idx for idx, label in enumerate(CATEGORY_LABELS_V3)}
id2category_v3 = {idx: label for label, idx in category2id_v3.items()}

print(f"Loaded {len(df_v3)} records")
print(f"Categories ({len(CATEGORY_LABELS_V3)}):")
for i, cat in enumerate(CATEGORY_LABELS_V3):
    print(f"  {i}: {cat}")

train_df_v3, val_df_v3 = train_test_split(
    df_v3,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_v3["urgency_label"],
)
print(f"\nTrain: {len(train_df_v3)}, Validation: {len(val_df_v3)}")

Loaded 1500 records
Categories (15):
  0: Abuse by adult in organisation
  1: Abuse by another young person
  2: Attendance / engagement
  3: Behaviour / conduct
  4: Bullying / peer conflict
  5: Exploitation / trafficking
  6: FGM / harmful practices
  7: Financial hardship
  8: Grooming
  9: Home issues / neglect
  10: Mental health / self-harm
  11: Online safety
  12: Physical safety
  13: Radicalisation / extremism
  14: Sexual abuse / assault

Train: 1200, Validation: 300


In [32]:
# Cell 23: Create datasets, model and train
from transformers import DistilBertModel
import torch.nn as nn

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, category2id, max_length=MAX_LENGTH):
        self.texts = dataframe["free_text"].tolist()
        self.urgency_labels = [label2id[label] for label in dataframe["urgency_label"]]
        self.category_labels = [category2id[label] for label in dataframe["category_label"]]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "urgency_label": torch.tensor(self.urgency_labels[idx], dtype=torch.long),
            "category_label": torch.tensor(self.category_labels[idx], dtype=torch.long),
        }

class MultiTaskSafeguardingModel(nn.Module):
    def __init__(self, model_name, num_urgency_labels, num_category_labels):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size
        self.urgency_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_urgency_labels),
        )
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_category_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.urgency_head(cls_output), self.category_head(cls_output)

# Create datasets and loaders
train_dataset_v3 = MultiTaskDataset(train_df_v3, tokenizer, label2id, category2id_v3)
val_dataset_v3 = MultiTaskDataset(val_df_v3, tokenizer, label2id, category2id_v3)
train_loader_v3 = DataLoader(train_dataset_v3, batch_size=BATCH_SIZE, shuffle=True)
val_loader_v3 = DataLoader(val_dataset_v3, batch_size=BATCH_SIZE, shuffle=False)

# Create fresh model
model_v3 = MultiTaskSafeguardingModel(
    model_name=MODEL_NAME,
    num_urgency_labels=len(URGENCY_LABELS),
    num_category_labels=len(CATEGORY_LABELS_V3),
)
model_v3.to(device)

print(f"Model ready on {device}")
print(f"Train batches: {len(train_loader_v3)}, Val batches: {len(val_loader_v3)}")
print(f"Urgency classes: {len(URGENCY_LABELS)}, Category classes: {len(CATEGORY_LABELS_V3)}")

Model ready on cuda
Train batches: 75, Val batches: 19
Urgency classes: 4, Category classes: 15


In [33]:
# Cell 24: Train multi-task v3 model
optimizer_v3 = torch.optim.AdamW(model_v3.parameters(), lr=LEARNING_RATE)
total_steps_v3 = len(train_loader_v3) * EPOCHS
warmup_steps_v3 = int(total_steps_v3 * WARMUP_RATIO)

scheduler_v3 = get_linear_schedule_with_warmup(
    optimizer_v3,
    num_warmup_steps=warmup_steps_v3,
    num_training_steps=total_steps_v3,
)

loss_fn = nn.CrossEntropyLoss()
URGENCY_WEIGHT = 0.5
CATEGORY_WEIGHT = 0.5

for epoch in range(EPOCHS):
    model_v3.train()
    running_loss = 0.0
    running_urg_loss = 0.0
    running_cat_loss = 0.0

    for batch in train_loader_v3:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        urgency_labels = batch["urgency_label"].to(device)
        category_labels = batch["category_label"].to(device)

        optimizer_v3.zero_grad()
        urgency_logits, category_logits = model_v3(input_ids, attention_mask)

        urg_loss = loss_fn(urgency_logits, urgency_labels)
        cat_loss = loss_fn(category_logits, category_labels)
        total_loss = (URGENCY_WEIGHT * urg_loss) + (CATEGORY_WEIGHT * cat_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v3.parameters(), max_norm=1.0)
        optimizer_v3.step()
        scheduler_v3.step()

        running_loss += total_loss.item()
        running_urg_loss += urg_loss.item()
        running_cat_loss += cat_loss.item()

    n_batches = len(train_loader_v3)

    # Validation
    model_v3.eval()
    urg_preds_v3, urg_true_v3 = [], []
    cat_preds_v3, cat_true_v3 = [], []

    with torch.no_grad():
        for batch in val_loader_v3:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            urgency_logits, category_logits = model_v3(input_ids, attention_mask)

            urg_preds_v3.extend(torch.argmax(urgency_logits, dim=-1).cpu().tolist())
            urg_true_v3.extend(batch["urgency_label"].tolist())
            cat_preds_v3.extend(torch.argmax(category_logits, dim=-1).cpu().tolist())
            cat_true_v3.extend(batch["category_label"].tolist())

    urg_acc = sum(p == l for p, l in zip(urg_preds_v3, urg_true_v3)) / len(urg_true_v3)
    cat_acc = sum(p == l for p, l in zip(cat_preds_v3, cat_true_v3)) / len(cat_true_v3)

    print(f"Epoch {epoch+1}/{EPOCHS}  |  Loss: {running_loss/n_batches:.4f} "
          f"(urg: {running_urg_loss/n_batches:.4f}, cat: {running_cat_loss/n_batches:.4f})  |  "
          f"Val urgency: {urg_acc:.4f}  Val category: {cat_acc:.4f}")

print("\nTraining complete.")

Epoch 1/10  |  Loss: 1.9983 (urg: 1.3202, cat: 2.6765)  |  Val urgency: 0.6033  Val category: 0.2633
Epoch 2/10  |  Loss: 1.5401 (urg: 0.8382, cat: 2.2420)  |  Val urgency: 0.6567  Val category: 0.4833
Epoch 3/10  |  Loss: 1.1267 (urg: 0.6208, cat: 1.6326)  |  Val urgency: 0.6767  Val category: 0.6333
Epoch 4/10  |  Loss: 0.8474 (urg: 0.4941, cat: 1.2008)  |  Val urgency: 0.7100  Val category: 0.6933
Epoch 5/10  |  Loss: 0.6386 (urg: 0.3657, cat: 0.9115)  |  Val urgency: 0.7300  Val category: 0.7900
Epoch 6/10  |  Loss: 0.4826 (urg: 0.2743, cat: 0.6910)  |  Val urgency: 0.7200  Val category: 0.8033
Epoch 7/10  |  Loss: 0.3706 (urg: 0.2049, cat: 0.5362)  |  Val urgency: 0.7033  Val category: 0.7967
Epoch 8/10  |  Loss: 0.2916 (urg: 0.1532, cat: 0.4300)  |  Val urgency: 0.7433  Val category: 0.8067
Epoch 9/10  |  Loss: 0.2389 (urg: 0.1254, cat: 0.3524)  |  Val urgency: 0.7367  Val category: 0.8067
Epoch 10/10  |  Loss: 0.2098 (urg: 0.1099, cat: 0.3097)  |  Val urgency: 0.7333  Val catego

In [34]:
# Cell 25: Detailed evaluation of v3 multi-task model
print("=== URGENCY CLASSIFICATION ===\n")
print(classification_report(
    urg_true_v3,
    urg_preds_v3,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\n=== CATEGORY CLASSIFICATION ===\n")
print(classification_report(
    cat_true_v3,
    cat_preds_v3,
    target_names=CATEGORY_LABELS_V3,
    digits=4,
))

=== URGENCY CLASSIFICATION ===

              precision    recall  f1-score   support

         Low     0.9559    0.8667    0.9091        75
      Medium     0.7468    0.7867    0.7662        75
        High     0.5402    0.6267    0.5802        75
    Critical     0.7424    0.6533    0.6950        75

    accuracy                         0.7333       300
   macro avg     0.7463    0.7333    0.7377       300
weighted avg     0.7463    0.7333    0.7377       300


=== CATEGORY CLASSIFICATION ===

                                precision    recall  f1-score   support

Abuse by adult in organisation     0.8750    0.9545    0.9130        22
 Abuse by another young person     0.7647    0.7647    0.7647        17
       Attendance / engagement     0.7727    0.7391    0.7556        23
           Behaviour / conduct     0.5789    0.6471    0.6111        17
      Bullying / peer conflict     0.7692    0.7143    0.7407        14
    Exploitation / trafficking     0.8095    0.7727    0.7907     

In [35]:
# Cell 26: Save v3 multi-task model
import os
import json

save_path_v3 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v2"
os.makedirs(save_path_v3, exist_ok=True)

torch.save(model_v3.state_dict(), os.path.join(save_path_v3, "model_state.pt"))
tokenizer.save_pretrained(save_path_v3)

config_v3 = {
    "model_name": MODEL_NAME,
    "num_urgency_labels": len(URGENCY_LABELS),
    "num_category_labels": len(CATEGORY_LABELS_V3),
    "urgency_labels": URGENCY_LABELS,
    "category_labels": CATEGORY_LABELS_V3,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "category2id": category2id_v3,
    "id2category": {str(k): v for k, v in id2category_v3.items()},
    "max_length": MAX_LENGTH,
    "val_urgency_accuracy": 0.7333,
    "val_category_accuracy": 0.8133,
}
with open(os.path.join(save_path_v3, "config.json"), "w") as f:
    json.dump(config_v3, f, indent=2)

print(f"Model saved to: {save_path_v3}")


Model saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v2


In [2]:
# Cell 27: Load v3 data and prepare weighted retraining
import torch.nn as nn
from transformers import DistilBertModel

DATA_PATH_V3 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v3.csv"
df_v3 = pd.read_csv(DATA_PATH_V3)

CATEGORY_LABELS_V3 = sorted(df_v3["category_label"].unique().tolist())
category2id_v3 = {label: idx for idx, label in enumerate(CATEGORY_LABELS_V3)}
id2category_v3 = {idx: label for label, idx in category2id_v3.items()}

train_df_v3, val_df_v3 = train_test_split(
    df_v3, test_size=0.2, random_state=RANDOM_SEED, stratify=df_v3["urgency_label"],
)

# Calculate class weights for urgency - penalise errors on Critical more heavily
# Inverse frequency weighting: rarer classes get higher weight
urgency_counts = train_df_v3["urgency_label"].value_counts()
total = len(train_df_v3)
urgency_weights = []
for label in URGENCY_LABELS:
    count = urgency_counts.get(label, 1)
    weight = total / (len(URGENCY_LABELS) * count)
    urgency_weights.append(weight)

# Additionally boost Critical by an extra factor of 1.5
# because under-triaging Critical cases is the most dangerous error
critical_idx = URGENCY_LABELS.index("Critical")
urgency_weights[critical_idx] *= 1.5

urgency_weight_tensor = torch.tensor(urgency_weights, dtype=torch.float).to(device)

print(f"Loaded {len(df_v3)} records. Train: {len(train_df_v3)}, Val: {len(val_df_v3)}")
print(f"\nUrgency class weights:")
for label, weight in zip(URGENCY_LABELS, urgency_weights):
    print(f"  {label}: {weight:.3f}")

Loaded 1500 records. Train: 1200, Val: 300

Urgency class weights:
  Low: 1.000
  Medium: 1.000
  High: 1.000
  Critical: 1.500


In [3]:
# Cell 28: Dataset, model and weighted training setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, category2id, max_length=MAX_LENGTH):
        self.texts = dataframe["free_text"].tolist()
        self.urgency_labels = [label2id[label] for label in dataframe["urgency_label"]]
        self.category_labels = [category2id[label] for label in dataframe["category_label"]]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "urgency_label": torch.tensor(self.urgency_labels[idx], dtype=torch.long),
            "category_label": torch.tensor(self.category_labels[idx], dtype=torch.long),
        }

class MultiTaskSafeguardingModel(nn.Module):
    def __init__(self, model_name, num_urgency_labels, num_category_labels):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size
        self.urgency_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_urgency_labels),
        )
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_category_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.urgency_head(cls_output), self.category_head(cls_output)

# Create datasets and loaders
train_dataset = MultiTaskDataset(train_df_v3, tokenizer, label2id, category2id_v3)
val_dataset = MultiTaskDataset(val_df_v3, tokenizer, label2id, category2id_v3)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Fresh model
model_v4 = MultiTaskSafeguardingModel(
    model_name=MODEL_NAME,
    num_urgency_labels=len(URGENCY_LABELS),
    num_category_labels=len(CATEGORY_LABELS_V3),
)
model_v4.to(device)

# Training config - 15 epochs this time
EPOCHS_V4 = 15

optimizer_v4 = torch.optim.AdamW(model_v4.parameters(), lr=LEARNING_RATE)
total_steps_v4 = len(train_loader) * EPOCHS_V4
warmup_steps_v4 = int(total_steps_v4 * WARMUP_RATIO)

scheduler_v4 = get_linear_schedule_with_warmup(
    optimizer_v4,
    num_warmup_steps=warmup_steps_v4,
    num_training_steps=total_steps_v4,
)

# Weighted loss for urgency, standard loss for category
urgency_loss_fn = nn.CrossEntropyLoss(weight=urgency_weight_tensor)
category_loss_fn = nn.CrossEntropyLoss()

print(f"Model ready on {device}")
print(f"Training for {EPOCHS_V4} epochs")
print(f"Total steps: {total_steps_v4}, Warmup: {warmup_steps_v4}")

Model ready on cuda
Training for 15 epochs
Total steps: 1125, Warmup: 112


In [4]:
# Cell 29: Train with weighted loss and 15 epochs
URGENCY_WEIGHT = 0.5
CATEGORY_WEIGHT = 0.5

best_urg_acc = 0.0
best_epoch = 0

for epoch in range(EPOCHS_V4):
    model_v4.train()
    running_loss = 0.0
    running_urg_loss = 0.0
    running_cat_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        urgency_labels = batch["urgency_label"].to(device)
        category_labels = batch["category_label"].to(device)

        optimizer_v4.zero_grad()
        urgency_logits, category_logits = model_v4(input_ids, attention_mask)

        urg_loss = urgency_loss_fn(urgency_logits, urgency_labels)
        cat_loss = category_loss_fn(category_logits, category_labels)
        total_loss = (URGENCY_WEIGHT * urg_loss) + (CATEGORY_WEIGHT * cat_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v4.parameters(), max_norm=1.0)
        optimizer_v4.step()
        scheduler_v4.step()

        running_loss += total_loss.item()
        running_urg_loss += urg_loss.item()
        running_cat_loss += cat_loss.item()

    n_batches = len(train_loader)

    # Validation
    model_v4.eval()
    urg_preds_v4, urg_true_v4 = [], []
    cat_preds_v4, cat_true_v4 = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            urgency_logits, category_logits = model_v4(input_ids, attention_mask)

            urg_preds_v4.extend(torch.argmax(urgency_logits, dim=-1).cpu().tolist())
            urg_true_v4.extend(batch["urgency_label"].tolist())
            cat_preds_v4.extend(torch.argmax(category_logits, dim=-1).cpu().tolist())
            cat_true_v4.extend(batch["category_label"].tolist())

    urg_acc = sum(p == l for p, l in zip(urg_preds_v4, urg_true_v4)) / len(urg_true_v4)
    cat_acc = sum(p == l for p, l in zip(cat_preds_v4, cat_true_v4)) / len(cat_true_v4)

    if urg_acc > best_urg_acc:
        best_urg_acc = urg_acc
        best_epoch = epoch + 1

    print(f"Epoch {epoch+1}/{EPOCHS_V4}  |  Loss: {running_loss/n_batches:.4f} "
          f"(urg: {running_urg_loss/n_batches:.4f}, cat: {running_cat_loss/n_batches:.4f})  |  "
          f"Val urgency: {urg_acc:.4f}  Val category: {cat_acc:.4f}")

print(f"\nTraining complete. Best urgency accuracy: {best_urg_acc:.4f} at epoch {best_epoch}")

Epoch 1/15  |  Loss: 2.0035 (urg: 1.3278, cat: 2.6791)  |  Val urgency: 0.4867  Val category: 0.2433
Epoch 2/15  |  Loss: 1.5815 (urg: 0.9017, cat: 2.2612)  |  Val urgency: 0.6500  Val category: 0.4633
Epoch 3/15  |  Loss: 1.0923 (urg: 0.6571, cat: 1.5275)  |  Val urgency: 0.6933  Val category: 0.6500
Epoch 4/15  |  Loss: 0.7708 (urg: 0.4904, cat: 1.0512)  |  Val urgency: 0.7333  Val category: 0.7633
Epoch 5/15  |  Loss: 0.5546 (urg: 0.3665, cat: 0.7426)  |  Val urgency: 0.6933  Val category: 0.8067
Epoch 6/15  |  Loss: 0.3892 (urg: 0.2648, cat: 0.5135)  |  Val urgency: 0.6967  Val category: 0.8033
Epoch 7/15  |  Loss: 0.2820 (urg: 0.1965, cat: 0.3676)  |  Val urgency: 0.7200  Val category: 0.8233
Epoch 8/15  |  Loss: 0.1997 (urg: 0.1431, cat: 0.2563)  |  Val urgency: 0.7200  Val category: 0.8167
Epoch 9/15  |  Loss: 0.1363 (urg: 0.0933, cat: 0.1793)  |  Val urgency: 0.7233  Val category: 0.8167
Epoch 10/15  |  Loss: 0.0964 (urg: 0.0702, cat: 0.1226)  |  Val urgency: 0.6867  Val catego

In [5]:
# Cell 30: Detailed evaluation of v4 model
print("=== URGENCY CLASSIFICATION ===\n")
print(classification_report(
    urg_true_v4,
    urg_preds_v4,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\nConfusion Matrix:")
cm = confusion_matrix(urg_true_v4, urg_preds_v4)
cm_df = pd.DataFrame(cm, index=URGENCY_LABELS, columns=URGENCY_LABELS)
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"
print(cm_df)

print("\n\n=== CATEGORY CLASSIFICATION ===\n")
print(classification_report(
    cat_true_v4,
    cat_preds_v4,
    target_names=CATEGORY_LABELS_V3,
    digits=4,
))

=== URGENCY CLASSIFICATION ===

              precision    recall  f1-score   support

         Low     0.9306    0.8933    0.9116        75
      Medium     0.7910    0.7067    0.7465        75
        High     0.4889    0.5867    0.5333        75
    Critical     0.6761    0.6400    0.6575        75

    accuracy                         0.7067       300
   macro avg     0.7216    0.7067    0.7122       300
weighted avg     0.7216    0.7067    0.7122       300


Confusion Matrix:
Predicted  Low  Medium  High  Critical
Actual                                
Low         67       6     2         0
Medium       5      53    17         0
High         0       8    44        23
Critical     0       0    27        48


=== CATEGORY CLASSIFICATION ===

                                precision    recall  f1-score   support

Abuse by adult in organisation     0.9167    1.0000    0.9565        22
 Abuse by another young person     0.7368    0.8235    0.7778        17
       Attendance / engageme

In [6]:
# Cell 31: Baseline model - TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report as sk_report

# Use the same train/val split from v3
X_train = train_df_v3["free_text"].tolist()
X_val = val_df_v3["free_text"].tolist()

y_train_urg = train_df_v3["urgency_label"].tolist()
y_val_urg = val_df_v3["urgency_label"].tolist()

y_train_cat = train_df_v3["category_label"].tolist()
y_val_cat = val_df_v3["category_label"].tolist()

# TF-IDF vectorisation
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# Urgency baseline
lr_urgency = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr_urgency.fit(X_train_tfidf, y_train_urg)
urg_preds_baseline = lr_urgency.predict(X_val_tfidf)

# Category baseline
lr_category = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr_category.fit(X_train_tfidf, y_train_cat)
cat_preds_baseline = lr_category.predict(X_val_tfidf)

print("=== BASELINE: TF-IDF + LOGISTIC REGRESSION ===")
print("\n--- Urgency Classification ---\n")
print(sk_report(y_val_urg, urg_preds_baseline, target_names=URGENCY_LABELS, digits=4))

print("\n--- Category Classification ---\n")
print(sk_report(y_val_cat, cat_preds_baseline, target_names=CATEGORY_LABELS_V3, digits=4))

=== BASELINE: TF-IDF + LOGISTIC REGRESSION ===

--- Urgency Classification ---

              precision    recall  f1-score   support

         Low     0.6000    0.7200    0.6545        75
      Medium     0.5254    0.4133    0.4627        75
        High     0.7432    0.7333    0.7383        75
    Critical     0.6234    0.6400    0.6316        75

    accuracy                         0.6267       300
   macro avg     0.6230    0.6267    0.6218       300
weighted avg     0.6230    0.6267    0.6218       300


--- Category Classification ---

                                precision    recall  f1-score   support

Abuse by adult in organisation     0.8000    0.7273    0.7619        22
 Abuse by another young person     0.6471    0.6471    0.6471        17
       Attendance / engagement     0.8000    0.6957    0.7442        23
           Behaviour / conduct     0.4500    0.5294    0.4865        17
      Bullying / peer conflict     0.6000    0.6429    0.6207        14
    Exploitation /

In [8]:
# Cell 32: Summary comparison table
comparison = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression", "Multi-task DistilBERT (v2)"],
    "Urgency Accuracy": [0.6267, 0.7333],
    "Urgency F1 (macro)": [0.6218, 0.7377],
    "Category Accuracy": [0.7100, 0.8133],
    "Category F1 (macro)": [0.7083, 0.8134],
    "Critical Recall": [0.6400, 0.6533],
})

print("=== MODEL COMPARISON SUMMARY ===\n")
print(comparison.to_string(index=False))

comparison.to_csv(
    r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\reports\model_comparison.csv",
    index=False,
)
print("\nSaved to reports/model_comparison.csv")


=== MODEL COMPARISON SUMMARY ===

                       Model  Urgency Accuracy  Urgency F1 (macro)  Category Accuracy  Category F1 (macro)  Critical Recall
TF-IDF + Logistic Regression            0.6267              0.6218             0.7100               0.7083           0.6400
  Multi-task DistilBERT (v2)            0.7333              0.7377             0.8133               0.8134           0.6533

Saved to reports/model_comparison.csv


In [2]:
# Cell 33: Load v4 data and retrain multi-task model
import torch.nn as nn
from transformers import DistilBertModel

DATA_PATH_V4 = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v4.csv"
df_v4 = pd.read_csv(DATA_PATH_V4)

CATEGORY_LABELS_V4 = sorted(df_v4["category_label"].unique().tolist())
category2id_v4 = {label: idx for idx, label in enumerate(CATEGORY_LABELS_V4)}
id2category_v4 = {idx: label for label, idx in category2id_v4.items()}

train_df_v4, val_df_v4 = train_test_split(
    df_v4, test_size=0.2, random_state=RANDOM_SEED, stratify=df_v4["urgency_label"],
)

print(f"Loaded {len(df_v4)} records. Train: {len(train_df_v4)}, Val: {len(val_df_v4)}")
print(f"Categories: {len(CATEGORY_LABELS_V4)}")

# Verify categories match v3
print(f"\nCategories:")
for i, cat in enumerate(CATEGORY_LABELS_V4):
    print(f"  {i}: {cat}")

Loaded 3500 records. Train: 2800, Val: 700
Categories: 15

Categories:
  0: Abuse by adult in organisation
  1: Abuse by another young person
  2: Attendance / engagement
  3: Behaviour / conduct
  4: Bullying / peer conflict
  5: Exploitation / trafficking
  6: FGM / harmful practices
  7: Financial hardship
  8: Grooming
  9: Home issues / neglect
  10: Mental health / self-harm
  11: Online safety
  12: Physical safety
  13: Radicalisation / extremism
  14: Sexual abuse / assault


In [5]:
# Cell 34: Setup and train
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, category2id, max_length=MAX_LENGTH):
        self.texts = dataframe["free_text"].tolist()
        self.urgency_labels = [label2id[label] for label in dataframe["urgency_label"]]
        self.category_labels = [category2id[label] for label in dataframe["category_label"]]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "urgency_label": torch.tensor(self.urgency_labels[idx], dtype=torch.long),
            "category_label": torch.tensor(self.category_labels[idx], dtype=torch.long),
        }

class MultiTaskSafeguardingModel(nn.Module):
    def __init__(self, model_name, num_urgency_labels, num_category_labels):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size
        self.urgency_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_size, num_urgency_labels),
        )
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_size, num_category_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.urgency_head(cls_output), self.category_head(cls_output)

train_dataset = MultiTaskDataset(train_df_v4, tokenizer, label2id, category2id_v4)
val_dataset = MultiTaskDataset(val_df_v4, tokenizer, label2id, category2id_v4)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_v5 = MultiTaskSafeguardingModel(
    model_name=MODEL_NAME,
    num_urgency_labels=len(URGENCY_LABELS),
    num_category_labels=len(CATEGORY_LABELS_V4),
)
model_v5.to(device)

EPOCHS_V5 = 10
optimizer = torch.optim.AdamW(model_v5.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS_V5
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps)
loss_fn = nn.CrossEntropyLoss()

print(f"Model ready on {device}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Training for {EPOCHS_V5} epochs, {total_steps} total steps")

for epoch in range(EPOCHS_V5):
    model_v5.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        urgency_labels = batch["urgency_label"].to(device)
        category_labels = batch["category_label"].to(device)

        optimizer.zero_grad()
        urg_logits, cat_logits = model_v5(input_ids, attention_mask)
        urg_loss = loss_fn(urg_logits, urgency_labels)
        cat_loss = loss_fn(cat_logits, category_labels)
        total_loss = 0.5 * urg_loss + 0.5 * cat_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v5.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += total_loss.item()

    # Validation
    model_v5.eval()
    urg_preds, urg_true, cat_preds, cat_true = [], [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            urg_logits, cat_logits = model_v5(input_ids, attention_mask)
            urg_preds.extend(torch.argmax(urg_logits, dim=-1).cpu().tolist())
            urg_true.extend(batch["urgency_label"].tolist())
            cat_preds.extend(torch.argmax(cat_logits, dim=-1).cpu().tolist())
            cat_true.extend(batch["category_label"].tolist())

    urg_acc = sum(p == l for p, l in zip(urg_preds, urg_true)) / len(urg_true)
    cat_acc = sum(p == l for p, l in zip(cat_preds, cat_true)) / len(cat_true)

    print(f"Epoch {epoch+1}/{EPOCHS_V5}  |  Loss: {running_loss/len(train_loader):.4f}  |  Val urgency: {urg_acc:.4f}  Val category: {cat_acc:.4f}")

print("\nTraining complete.")

Model ready on cuda
Train batches: 175, Val batches: 44
Training for 10 epochs, 1750 total steps
Epoch 1/10  |  Loss: 1.8595  |  Val urgency: 0.6171  Val category: 0.4714
Epoch 2/10  |  Loss: 1.0732  |  Val urgency: 0.7186  Val category: 0.7600
Epoch 3/10  |  Loss: 0.6315  |  Val urgency: 0.7429  Val category: 0.8143
Epoch 4/10  |  Loss: 0.4131  |  Val urgency: 0.7086  Val category: 0.8143
Epoch 5/10  |  Loss: 0.2742  |  Val urgency: 0.7700  Val category: 0.8429
Epoch 6/10  |  Loss: 0.1814  |  Val urgency: 0.7657  Val category: 0.8343
Epoch 7/10  |  Loss: 0.1138  |  Val urgency: 0.7686  Val category: 0.8329
Epoch 8/10  |  Loss: 0.0766  |  Val urgency: 0.7643  Val category: 0.8429
Epoch 9/10  |  Loss: 0.0543  |  Val urgency: 0.7714  Val category: 0.8457
Epoch 10/10  |  Loss: 0.0410  |  Val urgency: 0.7686  Val category: 0.8414

Training complete.


In [6]:
# Cell 35: Detailed evaluation of v5 model
print("=== URGENCY CLASSIFICATION ===\n")
print(classification_report(
    urg_true,
    urg_preds,
    target_names=URGENCY_LABELS,
    digits=4,
))

print("\nConfusion Matrix:")
cm = confusion_matrix(urg_true, urg_preds)
cm_df = pd.DataFrame(cm, index=URGENCY_LABELS, columns=URGENCY_LABELS)
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"
print(cm_df)

print("\n\n=== CATEGORY CLASSIFICATION ===\n")
print(classification_report(
    cat_true,
    cat_preds,
    target_names=CATEGORY_LABELS_V4,
    digits=4,
))

=== URGENCY CLASSIFICATION ===

              precision    recall  f1-score   support

         Low     0.9176    0.8914    0.9043       175
      Medium     0.7772    0.8171    0.7967       175
        High     0.6190    0.6686    0.6429       175
    Critical     0.7771    0.6971    0.7349       175

    accuracy                         0.7686       700
   macro avg     0.7727    0.7686    0.7697       700
weighted avg     0.7727    0.7686    0.7697       700


Confusion Matrix:
Predicted  Low  Medium  High  Critical
Actual                                
Low        156      17     1         1
Medium      13     143    19         0
High         1      23   117        34
Critical     0       1    52       122


=== CATEGORY CLASSIFICATION ===

                                precision    recall  f1-score   support

Abuse by adult in organisation     0.8864    0.9512    0.9176        41
 Abuse by another young person     0.6393    0.6964    0.6667        56
       Attendance / engageme

In [7]:
# Cell 36: Save v5 model
import os
import json

save_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v3"
os.makedirs(save_path, exist_ok=True)

torch.save(model_v5.state_dict(), os.path.join(save_path, "model_state.pt"))
tokenizer.save_pretrained(save_path)

config = {
    "model_name": MODEL_NAME,
    "num_urgency_labels": len(URGENCY_LABELS),
    "num_category_labels": len(CATEGORY_LABELS_V4),
    "urgency_labels": URGENCY_LABELS,
    "category_labels": CATEGORY_LABELS_V4,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "category2id": category2id_v4,
    "id2category": {str(k): v for k, v in id2category_v4.items()},
    "max_length": MAX_LENGTH,
    "val_urgency_accuracy": 0.7686,
    "val_category_accuracy": 0.8414,
    "training_records": len(train_df_v4),
}
with open(os.path.join(save_path, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to: {save_path}")

# Update comparison table
comparison = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression", "Multi-task DistilBERT v2 (1500 records)", "Multi-task DistilBERT v3 (3500 records)"],
    "Urgency Accuracy": [0.6267, 0.7333, 0.7686],
    "Urgency F1 (macro)": [0.6218, 0.7377, 0.7697],
    "Category Accuracy": [0.7100, 0.8133, 0.8414],
    "Category F1 (macro)": [0.7083, 0.8134, 0.8524],
    "Critical Recall": [0.6400, 0.6533, 0.6971],
})
print("\n=== MODEL COMPARISON ===\n")
print(comparison.to_string(index=False))

comparison.to_csv(
    r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\reports\model_comparison_v2.csv",
    index=False,
)
print("\nSaved to reports/model_comparison_v2.csv")

Model saved to: C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\models\multitask_v3

=== MODEL COMPARISON ===

                                  Model  Urgency Accuracy  Urgency F1 (macro)  Category Accuracy  Category F1 (macro)  Critical Recall
           TF-IDF + Logistic Regression            0.6267              0.6218             0.7100               0.7083           0.6400
Multi-task DistilBERT v2 (1500 records)            0.7333              0.7377             0.8133               0.8134           0.6533
Multi-task DistilBERT v3 (3500 records)            0.7686              0.7697             0.8414               0.8524           0.6971

Saved to reports/model_comparison_v2.csv
